In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN DEL DRIVER (EL CUERPO) ---
# Se define al principio para evitar abrir múltiples navegadores innecesariamente
options = Options()
options.add_argument("--headless")  # Ejecución invisible en el servidor Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
# La lista debe estar FUERA del bucle para acumular los datos de todas las páginas
datos_totales = []
paginas_objetivo = 5  # Límite para el ejercicio de laboratorio

try:
    # URL inicial del caso de estudio (E-commerce, Inmobiliario, etc.)
    driver.get("https://www.python.org/jobs/")

    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        # Espera explícita para asegurar que los elementos carguen
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "listing-item"))
        )

        # CAPTURA: Extraer elementos de la vista actual
        productos = driver.find_elements(By.CLASS_NAME, "listing-item")
        for p in productos:
            datos_totales.append(p.text)  # Guardamos en nuestra "mochila"

        # NAVEGACIÓN: Lógica para saltar a la siguiente página
        try:
            # Buscamos el botón "Siguiente" usando XPATH profesional
            # Nota: He corregido las comillas curvas ’ por comillas simples '
            boton_siguiente = driver.find_element(By.XPATH, "//li[@class='next']/a")

            # Hacemos clic y esperamos la carga de la nueva interfaz
            boton_siguiente.click()
            time.sleep(3)  # Pausa estratégica para el renderizado de JavaScript

        except Exception:
            print("Se ha llegado a la última página o el botón no es visible.")
            break  # Detiene el bucle si ya no hay más páginas

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO (CRUCIAL PARA DOCKER) ---
    # Esto evita que los procesos de Chrome 'zombis' agoten la RAM del laboratorio
    driver.quit()

--- Procesando Página 1 ---
Error detectado durante la ejecución: Message: 



In [2]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# ==========================================================
# 1. MOTOR DE NAVEGACIÓN (Configuración para Docker)
# ==========================================================

def iniciar_navegador():
    options = Options()
    # Esta ruta es la que configuramos en tu Dockerfile
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ==========================================================
# 2. CONFIGURACIÓN DE LA BÚSQUEDA
# ==========================================================

# url del sitio
URL_OBJETIVO = "https://www.python.org/jobs/"

# Estos selectores son específicos para la lista de empleos de Python

SELECTOR_CONTENEDOR = "li"
SELECTOR_DATO_A     = "span.listing-job-type"         # Ejemplo: "Back end"
SELECTOR_DATO_B     = "span.listing-company-category"  # Ejemplo: "Developer"

# ==========================================================
# 3. EJECUCIÓN DEL SCRAPING
# ==========================================================

driver = iniciar_navegador()
dataset_final = []

try:
    print(f" Conectando a: {URL_OBJETIVO}...")
    driver.get(URL_OBJETIVO)

    # Esperamos 5 segundos para que cargue bien la lista

    time.sleep(5)

    # Capturamos los elementos de la lista

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f" Se encontraron {len(elementos)} líneas analizadas.")

    for item in elementos:
        try:
            # Intentamos extraer el texto de las etiquetas
            tipo_trabajo = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A).text
            categoria = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            # Guardamos los datos en la lista

            dataset_final.append({

                "tipo_puesto": tipo_trabajo.strip(),
                "categoria": categoria.strip(),
                "fecha_extraccion": time.strftime("%Y-%m-%d %H:%M"),
                "status": 1.0
            })
        except:
            # Si el 'li' no tiene los datos (es publicidad o menú), lo saltamos
            continue
    # ==========================================================

    # 4. GUARDAR Y MOSTRAR RESULTADOS

    # ==========================================================

    if dataset_final:
        df = pd.DataFrame(dataset_final)
        nombre_archivo = "scraping_python_jobs.csv"
        df.to_csv(nombre_archivo, index=False)
        print(f" ¡Éxito! Archivo generado: {nombre_archivo}")
        print("\n--- Muestra de los datos extraídos ---")
        print(df.head())
    else:
        print(" No se capturaron datos. Revisa la URL o los selectores.")

except Exception as e:
    print(f" Error crítico: {e}")

finally:
    driver.quit()
    print(" Navegador cerrado correctamente.")

 Conectando a: https://www.python.org/jobs/...
 Se encontraron 146 líneas analizadas.
 ¡Éxito! Archivo generado: scraping_python_jobs.csv

--- Muestra de los datos extraídos ---
                                         tipo_puesto             categoria  \
0                             Cloud, Systems, DevOps  Developer / Engineer   
1                   Back end, Database, Testing, Web  Developer / Engineer   
2                                           Back end  Developer / Engineer   
3  Back end, Big Data, Cloud, Database, Machine L...  Developer / Engineer   
4                         Big Data, Image Processing  Developer / Engineer   

   fecha_extraccion  status  
0  2026-04-01 03:04     1.0  
1  2026-04-01 03:04     1.0  
2  2026-04-01 03:04     1.0  
3  2026-04-01 03:04     1.0  
4  2026-04-01 03:04     1.0  
 Navegador cerrado correctamente.
